# ParquetScoresDB API tour

`ParquetScoresDB` (`perch_hoplite/db/parquet_scores_impl.py`) is a
DuckDB + Parquet backend for wide per-window class-score outputs (e.g.
classifier scores over 10-10,000 classes per clip), as an alternative to
`SQLiteUSearchDB`'s single-embedding-per-window model. This notebook is a
guided tour of its core API, organized around three tasks:

1. **Adding scores** -- deployments, recordings, and score matrices, in one
   call or split across several ("chunked") calls.
2. **Selecting stratified scores/detections** -- the top/random/percentile
   -binned selection queries, stratified by deployment, arbitrary
   deployment fields, and/or date.
3. **Retrieving entire blocks of scores** -- unfiltered or filtered bulk
   reads, as arrays or as a DataFrame, optionally merged with
   deployment/recording metadata.

See `PARQUET_DUCKDB_PLAN.md` and `PARQUET_CHUNKING_IMPLEMENTATION_PLAN.md`
in this directory for the design discussion behind these choices.

In [1]:
import datetime as dt
import shutil
import tempfile

import numpy as np
import pandas as pd
from ml_collections import config_dict
from perch_hoplite.db import parquet_scores_impl as psi

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

DB_PATH = tempfile.mkdtemp(prefix='parquet_scores_tour_')
print(f'DB_PATH = {DB_PATH}')

DB_PATH = /var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr/T/parquet_scores_tour_qrft8qbx


## 1. Adding scores

### 1a. Create the database with a fixed class vocabulary

The class vocabulary is set once, at creation time, and is immutable for
the life of the database (every deployment's scores must come from the
same classifier/class list).

In [2]:
CLASS_LABELS = ('crow', 'robin', 'sparrow', 'silence')

db = psi.ParquetScoresDB.create(DB_PATH, class_labels=CLASS_LABELS)
print(db.classes)

('crow', 'robin', 'sparrow', 'silence')


### 1b. Register deployments and recordings

`deployments` are sites (with optional lat/lon and arbitrary extra
columns, e.g. `management_unit` below); `recordings` are individual audio
files belonging to a deployment. Both `insert_deployment`/
`insert_recording` are upserts keyed on `(name, project)` /
`(filename, deployment_id)` respectively.

In [3]:
site_a = db.insert_deployment(
    name='site_a',
    project='demo_project',
    latitude=37.87,
    longitude=-122.27,
    management_unit='unit_north',  # arbitrary extra column
)
site_b = db.insert_deployment(
    name='site_b',
    project='demo_project',
    latitude=37.90,
    longitude=-122.30,
    management_unit='unit_south',
)
print(f'site_a={site_a}, site_b={site_b}')

rng = np.random.default_rng(0)


def register_recordings(deployment_id, n, start_date):
  """Register `n` daily recordings for a deployment, returning their ids."""
  return [
      db.insert_recording(
          filename=f'dep{deployment_id}_day{i:03d}.wav',
          deployment_id=deployment_id,
          datetime=start_date + dt.timedelta(days=i),
      )
      for i in range(n)
  ]


site_a_recordings = register_recordings(
    site_a, 5, dt.datetime(2024, 1, 1, tzinfo=dt.timezone.utc)
)
site_b_recordings = register_recordings(
    site_b, 5, dt.datetime(2024, 1, 1, tzinfo=dt.timezone.utc)
)
print(f'site_a recordings: {site_a_recordings}')
print(f'site_b recordings: {site_b_recordings}')

site_a=1, site_b=2
site_a recordings: [1, 2, 3, 4, 5]
site_b recordings: [6, 7, 8, 9, 10]


### 1c. Write scores: one call per deployment (the common case)

`insert_deployment_scores()` is the only way to write scores -- there is
no per-window insertion, since Parquet isn't designed for single-row
appends. Each call writes recording ids / start & end offsets / a
`[n_windows, n_classes]` score matrix together, and the whole matrix for a
deployment is often written in one call, right after a single inference
pass over that deployment.

Here each recording gets 4 non-overlapping 3-second clips.

In [4]:
def make_synthetic_scores(recording_ids, clips_per_recording, n_classes, rng):
  """Build (recording_ids, start_times, end_times, scores) for synthetic clips."""
  out_recording_ids, start_times, end_times = [], [], []
  for recording_id in recording_ids:
    for clip_idx in range(clips_per_recording):
      out_recording_ids.append(recording_id)
      start_times.append(clip_idx * 3.0)
      end_times.append((clip_idx + 1) * 3.0)
  n = len(out_recording_ids)
  scores = rng.normal(loc=-2.0, scale=1.0, size=(n, n_classes)).astype(
      np.float32
  )
  return out_recording_ids, start_times, end_times, scores

recording_ids, start_times, end_times, scores = make_synthetic_scores(
    site_a_recordings, clips_per_recording=4, n_classes=len(CLASS_LABELS), rng=rng
)
db.insert_deployment_scores(site_a, recording_ids, start_times, end_times, scores)

recording_ids, start_times, end_times, scores = make_synthetic_scores(
    site_b_recordings, clips_per_recording=4, n_classes=len(CLASS_LABELS), rng=rng
)
db.insert_deployment_scores(site_b, recording_ids, start_times, end_times, scores)

print(f'site_a windows: {db.count_windows(deployment_id=site_a)}')
print(f'site_b windows: {db.count_windows(deployment_id=site_b)}')
print(f'total windows: {db.count_windows()}')

site_a windows: 20
site_b windows: 20
total windows: 40


### 1d. Writing scores across multiple calls ("chunking")

A deployment doesn't have to be written all at once: each call to
`insert_deployment_scores()` writes one new Parquet "chunk" file, and a
deployment's scores can live across several chunk files -- useful for
bounding peak memory on a very large deployment, or when scoring happens
incrementally (e.g. as new recordings arrive). Every read method
transparently unions all of a deployment's chunks.

Register a third deployment and write its recordings' scores in two
separate calls, split by recording.

In [5]:
site_c = db.insert_deployment(name='site_c', project='demo_project')
site_c_recordings = register_recordings(
    site_c, 4, dt.datetime(2024, 1, 1, tzinfo=dt.timezone.utc)
)

first_half, second_half = site_c_recordings[:2], site_c_recordings[2:]

r_ids, s_times, e_times, sc = make_synthetic_scores(
    first_half, clips_per_recording=4, n_classes=len(CLASS_LABELS), rng=rng
)
db.insert_deployment_scores(site_c, r_ids, s_times, e_times, sc)

r_ids, s_times, e_times, sc = make_synthetic_scores(
    second_half, clips_per_recording=4, n_classes=len(CLASS_LABELS), rng=rng
)
db.insert_deployment_scores(site_c, r_ids, s_times, e_times, sc)

print(f'site_c chunk files: {len(db.deployment_chunk_paths(site_c))}')
print(f'site_c windows (both chunks, queried together): {db.count_windows(deployment_id=site_c)}')

site_c chunk files: 2
site_c windows (both chunks, queried together): 16


### 1e. Re-inserting scores for already-scored recordings

`handle_existing_recordings` controls what happens when a batch includes a
recording that some earlier call already wrote scores for -- checked
**per recording**, not per row: a recording's scores are always written
together, so there's no partial-recording state.

- `'error'` (default): raise if any recording in the batch already has
  scores.
- `'skip'`: drop already-scored recordings from the batch; write only the
  genuinely-new ones.
- `'overwrite'`: replace already-scored recordings' rows in place within
  their existing chunk file(s) (their chunk location doesn't change); only
  genuinely-new recordings get a new chunk.

**Important**: `'overwrite'` replaces *all* of a recording's existing rows
with whatever rows for that recording are in the new batch -- it is not a
per-row upsert. If a recording has 4 scored windows and you re-insert it
with only 1 window's worth of scores, the other 3 windows are dropped, not
preserved. Always pass a recording's complete set of windows when
overwriting it.

In [6]:
try:
  db.insert_deployment_scores(
      site_a, [site_a_recordings[0]], [0.0], [3.0],
      np.zeros((1, len(CLASS_LABELS)), dtype=np.float32),
  )
except ValueError as e:
  print(f'default (error): {e}')

# Correcting recording 0's scores: pass ALL 4 of its windows, not just one,
# since 'overwrite' replaces the recording's entire row set.
corrected_recording_ids, corrected_starts, corrected_ends, _ = (
    make_synthetic_scores(
        [site_a_recordings[0]], clips_per_recording=4,
        n_classes=len(CLASS_LABELS), rng=rng,
    )
)
corrected_scores = np.full((4, len(CLASS_LABELS)), -5.0, dtype=np.float32)
db.insert_deployment_scores(
    site_a, corrected_recording_ids, corrected_starts, corrected_ends,
    corrected_scores, handle_existing_recordings='overwrite',
)
print(
    'after overwrite:',
    db.get_scores_batch(
        site_a, corrected_recording_ids, corrected_starts, corrected_ends
    ),
)

default (error): The following recordings already have scores written: [1]. Pass handle_existing_recordings='skip' to skip them, or ='overwrite' to replace their scores in place.
after overwrite: [[-5. -5. -5. -5.]
 [-5. -5. -5. -5.]
 [-5. -5. -5. -5.]
 [-5. -5. -5. -5.]]


## 2. Selecting stratified scores/detections

`select_scores()` composes a **selection mode** (`'top'`, `'random'`,
`'binned'`) with optional, independent **stratification** (by deployment,
by arbitrary `deployments` field(s), by date bin, or any combination) and
a set of filters -- all in a single DuckDB query, no Python loop over
strata.

### 2a. Top-N scores for one class, globally

In [7]:
top_crow = db.select_scores('crow', selection='top', n=5)
top_crow

,deployment_id,deployment_name,recording_id,recording_filename,recording_datetime,start_time,end_time,score
0,3,site_c,11,dep3_day000.wav,2024-01-01 00:00:00+00:00,3.0,6.0,-0.183472
1,1,site_a,4,dep1_day003.wav,2024-01-04 00:00:00+00:00,0.0,3.0,-0.198364
2,3,site_c,11,dep3_day000.wav,2024-01-01 00:00:00+00:00,6.0,9.0,-0.352539
3,2,site_b,8,dep2_day002.wav,2024-01-03 00:00:00+00:00,3.0,6.0,-0.579102
4,2,site_b,9,dep2_day003.wav,2024-01-04 00:00:00+00:00,6.0,9.0,-0.597656


### 2b. Stratified by deployment: top-N *per deployment*

In [8]:
top_crow_by_deployment = db.select_scores(
    'crow', selection='top', n=2, by_deployment=True
)
top_crow_by_deployment[['deployment_id', 'deployment_name', 'score', 'stratum']]

,deployment_id,deployment_name,score,stratum
0,1,site_a,-0.198364,1
1,1,site_a,-0.996094,1
2,2,site_b,-0.579102,2
3,2,site_b,-0.597656,2
4,3,site_c,-0.183472,3
5,3,site_c,-0.352539,3


### 2c. Stratified by an arbitrary deployment field

`by_deployment_fields` generalizes `by_deployment` to any `deployments`
column, built-in or extra (here, the `management_unit` extra column
registered above). `by_deployment=True` is exactly
`by_deployment_fields=['id']`.

In [9]:
top_crow_by_unit = db.select_scores(
    'crow', selection='top', n=3, by_deployment_fields=['management_unit']
)
top_crow_by_unit[['management_unit', 'deployment_id', 'score', 'stratum']]

,management_unit,deployment_id,score,stratum
0,unit_north,1,-0.198364,unit_north
1,unit_north,1,-0.996094,unit_north
2,unit_north,1,-1.096680,unit_north
3,unit_south,2,-0.579102,unit_south
4,unit_south,2,-0.597656,unit_south
5,unit_south,2,-1.126953,unit_south
6,None,3,-0.183472,None
7,None,3,-0.352539,None
8,None,3,-0.913086,None


### 2d. Stratified by date bin, and by deployment + date bin together

`date_bins` accepts `'day'`/`'week'` (calendar buckets) or an explicit
`(bin_start, label)` sequence. It composes with `by_deployment`/
`by_deployment_fields`.

In [10]:
top_crow_by_unit_and_week = db.select_scores(
    'crow',
    selection='top',
    n=1,
    by_deployment_fields=['management_unit'],
    date_bins='week',
)
top_crow_by_unit_and_week[
    ['management_unit', 'date_bin_label', 'score', 'stratum']
]

,management_unit,date_bin_label,score,stratum
0,unit_north,2024-01-01 00:00:00+00:00,-0.198364,"{'management_unit': 'unit_north', 'date_bin': ..."
1,unit_south,2024-01-01 00:00:00+00:00,-0.579102,"{'management_unit': 'unit_south', 'date_bin': ..."
2,None,2024-01-01 00:00:00+00:00,-0.183472,"{'management_unit': None, 'date_bin': 2024-01-..."


### 2e. Random sampling and percentile-binned selection

`selection='random'` draws uniformly (optionally with a `random_seed` for
reproducibility); `selection='binned'` takes the top-`n` rows from each
percentile bin of the (stratum-local) score distribution.

In [11]:
random_sample = db.select_scores(
    'sparrow', selection='random', n=5, random_seed=0
)
print('random sample scores:', sorted(random_sample['score'].round(2)))

binned = db.select_scores(
    'sparrow',
    selection='binned',
    n=1,
    bin_edges=[0, 0.5, 0.75, 0.875, 1],  # 4 percentile bins
)
binned[['score', 'bin_idx']]

random sample scores: [-2.869999885559082, -2.5299999713897705, -2.4800000190734863, -1.909999966621399, -1.840000033378601]


,score,bin_idx
0,-2.287109,0
1,-1.642578,1
2,-1.215820,2
3,-0.177979,3


### 2f. Filters: deployments, recordings, score range, date range

`deployments_filter` uses the deployments table's own columns (`id`,
`name`, `project`, ..., or extras) -- NOT `deployment_id`, which is only a
foreign key on `recordings`/`detections`.

In [12]:
filtered = db.select_scores(
    'crow',
    selection='top',
    n=10,
    deployments_filter=config_dict.create(eq=dict(management_unit='unit_north')),
    start_date=dt.datetime(2024, 1, 2, tzinfo=dt.timezone.utc),
    min_score=-3.0,
)
print(f'{len(filtered)} rows matched (unit_north, from day 2 on, score >= -3.0)')

10 rows matched (unit_north, from day 2 on, score >= -3.0)


### 2g. Detections: thresholding and stratified selection

`add_detections_from_threshold()` scans the score Parquet files directly
and bulk-inserts qualifying `(recording, offsets, class, score)` tuples
into the small `detections` table. `select_detections()` then mirrors
`select_scores()`'s selection/stratification API over that table.

In [13]:
detection_ids = db.add_detections_from_threshold(
    threshold=-1.0, class_name='crow', provenance='demo_threshold_scan'
)
print(f'{len(detection_ids)} detections added for crow >= -1.0')

top_detections_by_deployment = db.select_detections(
    class_name='crow', selection='top', n=2, by_deployment=True
)
top_detections_by_deployment[
    ['deployment_id', 'recording_filename', 'start_time', 'score']
]

7 detections added for crow >= -1.0


,deployment_id,recording_filename,start_time,score
0,1,dep1_day003.wav,0.0,-0.198364
1,1,dep1_day004.wav,6.0,-0.996094
2,2,dep2_day002.wav,3.0,-0.579102
3,2,dep2_day003.wav,6.0,-0.597656
4,3,dep3_day000.wav,3.0,-0.183472
5,3,dep3_day000.wav,6.0,-0.352539


### 2h. Sampling effort: how much data exists per stratum

`sampling_effort()` mirrors `select_scores()`'s stratification args, but
reports **window counts** rather than score values -- and crucially,
every stratum in the grid gets a row, even ones with zero windows, so
coverage gaps are visible rather than silently absent.

In [14]:
effort_by_deployment_and_week = db.sampling_effort(
    by_deployment=True, date_bins='week'
)
effort_by_deployment_and_week

,deployment_id,deployment_name,date_bin_label,n_windows
0,1,site_a,2024-01-01 00:00:00+00,20
1,2,site_b,2024-01-01 00:00:00+00,20
2,3,site_c,2024-01-01 00:00:00+00,16


## 3. Retrieving entire blocks of scores

For bulk/unfiltered or coarsely-filtered reads (rather than a ranked
selection), use `get_scores()` (filtered scan, columnar `ScoreBatch`
result) or `get_scores_batch()` (fetch specific known windows).

### 3a. `get_scores()`: the columnar analog of `get_all_windows()`

Returns a `ScoreBatch` -- parallel `deployment_ids`/`recording_ids`/
`start_times`/`end_times` arrays plus one `[N, n_classes]` `scores` array
-- rather than a list of per-window objects, matching the array-based
shape the rest of this API already uses.

In [15]:
all_scores = db.get_scores()
print(f'{len(all_scores)} windows, scores shape = {all_scores.scores.shape}')
print(f'class_labels = {all_scores.class_labels}')

# Filtered: a specific deployment, within a date range.
site_a_scores = db.get_scores(
    deployments_filter=config_dict.create(eq=dict(name='site_a')),
    start_date=dt.datetime(2024, 1, 3, tzinfo=dt.timezone.utc),
)
print(f'site_a from day 3 on: {len(site_a_scores)} windows')

56 windows, scores shape = (56, 4)
class_labels = ('crow', 'robin', 'sparrow', 'silence')
site_a from day 3 on: 12 windows


### 3b. Converting to a DataFrame, and merging in metadata

`ScoreBatch.to_pandas()` gives one row per window, one column per class.
`add_deployment_metadata()`/`add_recording_metadata()` are standalone
helpers (not tied to `ScoreBatch` specifically -- they work on any frame
with the right foreign-key column, including `select_scores()`'s output)
that left-join in the full `deployments`/`recordings` tables, including
any extra columns, with the newly-added columns placed first.

In [16]:
df = all_scores.to_pandas()
df = psi.add_deployment_metadata(db, df)
df = psi.add_recording_metadata(db, df)
df.head()

/Users/sala6791/perch-hoplite/.venv/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()
/Users/sala6791/perch-hoplite/.venv/lib/python3.11/site-packages/pandas/io/formats/format.py:1458: RuntimeWarning: overflow encountered in cast
  has_large_values = (abs_vals > 1e6).any()


,filename,datetime,score_chunk_path,deployment_name,project,latitude,longitude,management_unit,deployment_id,recording_id,start_time,end_time,crow,robin,sparrow,silence
0,dep1_day001.wav,2024-01-02 00:00:00+00:00,/var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr...,site_a,demo_project,37.87,-122.27,unit_north,1,2,0.0,3.0,-2.544922,-2.316406,-1.587891,-0.957520
1,dep1_day001.wav,2024-01-02 00:00:00+00:00,/var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr...,site_a,demo_project,37.87,-122.27,unit_north,1,2,3.0,6.0,-2.128906,-0.633301,-2.666016,-1.648438
2,dep1_day001.wav,2024-01-02 00:00:00+00:00,/var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr...,site_a,demo_project,37.87,-122.27,unit_north,1,2,6.0,9.0,-1.096680,-1.906250,-2.744141,-2.921875
3,dep1_day001.wav,2024-01-02 00:00:00+00:00,/var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr...,site_a,demo_project,37.87,-122.27,unit_north,1,2,9.0,12.0,-2.457031,-1.780273,-3.009766,-2.208984
4,dep1_day002.wav,2024-01-03 00:00:00+00:00,/var/folders/_1/clc139q504qfy1s2zl7qf9h00000gr...,site_a,demo_project,37.87,-122.27,unit_north,1,3,0.0,3.0,-2.160156,-1.458984,-1.785156,-1.644531


### 3c. `get_scores_batch()`: fetch specific known windows

Given explicit `(recording_id, start_time, end_time)` keys (e.g. the
windows around a detection), fetch just their score vectors -- pruned to
only the chunk file(s) that actually contain the requested recordings.

In [17]:
batch = db.get_scores_batch(
    deployment_id=site_a,
    recording_ids=[site_a_recordings[0], site_a_recordings[0]],
    start_times=[0.0, 3.0],
    end_times=[3.0, 6.0],
)
print(f'batch shape: {batch.shape}')
print(batch)

batch shape: (2, 4)
[[-5. -5. -5. -5.]
 [-5. -5. -5. -5.]]


### 3d. Removing a deployment

`remove_deployment()` deletes the deployment's metadata rows (cascading
to recordings/annotations/detections) **and** all of its chunk Parquet
files outright -- an exact delete, not an orphaning of Parquet rows.

In [18]:
db.remove_deployment(site_c)
print(f'deployments remaining: {[d.name for d in db.get_all_deployments()]}')
print(f'total windows after removing site_c: {db.count_windows()}')

deployments remaining: ['site_a', 'site_b']
total windows after removing site_c: 40


## Cleanup

In [19]:
db.commit()
db.db.close()
shutil.rmtree(DB_PATH, ignore_errors=True)